# 02.5 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/02-regression/05-multiple-regression/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest
import pandas as pd
import numpy as np

cust20 = pd.read_csv(data_path("shopwave_customers_small.csv"))
cust20.head()

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
y = cust20["spend_next_month"]
feats = ["visits_last_month", "items_last_month", "days_since_last"]
three = LinearRegression().fit(cust20[feats], y)

## Exercise 1

30 days: $38.1 + 113.2 + 30.6 - 21.0 = 160.9$. 60 days: $38.1 + 113.2 + 30.6 - 42.0 = 139.9$. Difference **£21**, which is $30 \times 0.70$: thirty more silent days, at −£0.70 each. Only the `days_since_last` coefficient is involved because the other two inputs did not change.

In [2]:
new = pd.DataFrame({"visits_last_month": [9, 9], "items_last_month": [4, 4], "days_since_last": [30, 60]})
p = three.predict(new); print(p.round(1), "difference", round(p[0] - p[1], 1))

[160.9 139.8] difference 21.1


## Exercise 2

In [3]:
from itertools import combinations
rows = []
for r in [1, 2, 3]:
    for subset in combinations(feats, r):
        m = LinearRegression().fit(cust20[list(subset)], y); p = m.predict(cust20[list(subset)])
        rows.append({"features": " + ".join(s.replace("_last_month", "").replace("_since_last", "") for s in subset),
                     "R²": round(r2_score(y, p), 3), "MAE": round(mean_absolute_error(y, p), 2)})
pd.DataFrame(rows).sort_values("R²", ascending=False)

,features,R²,MAE
6,visits + items + days,0.983,7.80
4,visits + days,0.977,8.23
0,visits,0.962,10.65
3,visits + items,0.962,10.82
5,items + days,0.934,14.01
1,items,0.869,19.86
2,days,0.115,52.82


Visits is the best single feature. Adding items to visits does almost nothing (they carry the same information); adding days helps clearly. The best pair is visits + days, and all three is only marginally better than that.

## Exercise 3

In [4]:
rng = np.random.default_rng(1)
X_noise = cust20[feats].assign(noise=rng.normal(size=len(cust20)))
m = LinearRegression().fit(X_noise, y)
print(f"R² without noise: {r2_score(y, three.predict(cust20[feats])):.4f}")
print(f"R² with noise:    {r2_score(y, m.predict(X_noise)):.4f}   noise coefficient {m.coef_[-1]:.2f}")

R² without noise: 0.9827
R² with noise:    0.9850   noise coefficient 6.39


R² goes *up*, slightly, for a column of random numbers. Training R² rewards every extra column, useful or not. The only honest score is on rows the model never saw (02.6, 04.1).

## Exercise 4

In [5]:
import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail
raw = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")

def clean_retail(df):
    out = df.copy()
    out = out[~out["Invoice"].astype(str).str.startswith("C")]
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)]
    out = out.dropna(subset=["Customer ID"]).drop_duplicates()
    out["Customer ID"] = out["Customer ID"].astype(int)
    out["Revenue"] = out["Quantity"] * out["Price"]
    return out
sales = clean_retail(raw)
cutoff = pd.Timestamp("2011-09-01")
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]
cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
    products=("StockCode", "nunique"), items=("Quantity", "sum"),
    first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["avg_order"] = cust["spend"] / cust["orders"]
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
cust["spend_next_quarter"] = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)

In [6]:
six = ["orders", "spend", "products", "avg_order", "tenure_days", "days_since_last"]
y_real = cust["spend_next_quarter"]
a = LinearRegression().fit(cust[six], y_real)
b = LinearRegression().fit(cust[six + ["items"]], y_real)
print(f"corr(spend, items) = {cust['spend'].corr(cust['items']):.3f}")
print(f"coefficient of spend:  without items {a.coef_[1]:.3f}   with items {b.coef_[1]:.3f}   (items coef {b.coef_[-1]:.4f})")
print(f"R²: {r2_score(y_real, a.predict(cust[six])):.4f} → {r2_score(y_real, b.predict(cust[six + ['items']])):.4f}")

corr(spend, items) = 0.918
coefficient of spend:  without items 0.616   with items 0.869   (items coef -0.4803)
R²: 0.7203 → 0.7501


`spend` and `items` are almost the same column (correlation 0.92), so adding `items` makes the model split one effect across two coefficients: `spend`'s coefficient jumps from 0.62 to 0.87 and `items` goes *negative*, while R² moves only a few points. The predictions are stable; the *explanation* is not. When you need to interpret coefficients, keep one feature per idea.